In [1]:
!pip install datasets gensim torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 45.6 MB/s eta 0:00:00


In [2]:
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset
from gensim.models import Word2Vec
from torch.utils.data import Dataset, DataLoader

In [3]:
dataset = load_dataset(
    "agentlans/high-quality-english-sentences",
    split="train[:1000]"
)

sentences = [row["text"] for row in dataset]

sentences[:5]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.30k [00:00<?, ?B/s]

train.txt.gz:   0%|          | 0.00/85.5M [00:00<?, ?B/s]

test.txt.gz:   0%|          | 0.00/9.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1534699 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/170522 [00:00<?, ? examples/s]

['Soon we dropped into a living forest, where cold-tolerant evergreens and boreal animals still evoke the Canadian heritage of an ecosystem pushed south by glaciers 20,000 years ago.',
 'Annual population growth rate (2011 est., CIA World Factbook): 1.284%.',
 'This has led to the recent banning of Neonics in the EU, however the US and Canada are still using this chemical pesticide.',
 "In addition, these colors weren't confined to a province but rather irregularly scattered across various regions over all of China.",
 'A family member or a support person may stay with a patient during recovery.']

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


clean_sentences = [clean_text(sentence) for sentence in sentences]

# Eliminamos frases muy cortas
clean_sentences = [s for s in clean_sentences if len(s.split()) >= 5]

clean_sentences[:5]

['soon we dropped into a living forest where coldtolerant evergreens and boreal animals still evoke the canadian heritage of an ecosystem pushed south by glaciers 20000 years ago',
 'annual population growth rate 2011 est cia world factbook 1284',
 'this has led to the recent banning of neonics in the eu however the us and canada are still using this chemical pesticide',
 'in addition these colors werent confined to a province but rather irregularly scattered across various regions over all of china',
 'a family member or a support person may stay with a patient during recovery']

# Crear vocabulario

In [5]:
tokenized_sentences = [sentence.split() for sentence in clean_sentences]

word_counts = {}

for sentence in tokenized_sentences:
    for word in sentence:
        word_counts[word] = word_counts.get(word, 0) + 1

vocab = {
    "<PAD>": 0,
    "<OOV>": 1
}

for word in word_counts:
    vocab[word] = len(vocab)

index_to_word = {index: word for word, index in vocab.items()}

vocab_size = len(vocab)

print("Tamaño del vocabulario:", vocab_size)

Tamaño del vocabulario: 6194


# Convertir frases a números

In [6]:
def text_to_sequence(sentence):
    return [vocab.get(word, vocab["<OOV>"]) for word in sentence.split()]


sequences = [text_to_sequence(sentence) for sentence in clean_sentences]

sequences[:2]


[[2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29],
 [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]]

# Crear secuencias para predecir la siguiente palabra

In [7]:
input_sequences = []

for sequence in sequences:
    for i in range(2, len(sequence)):
        input_sequences.append(sequence[:i+1])

print("Cantidad de secuencias:", len(input_sequences))

Cantidad de secuencias: 19566


In [8]:
max_sequence_len = max(len(seq) for seq in input_sequences)

print("Longitud máxima:", max_sequence_len)

Longitud máxima: 89


In [9]:
def pad_sequence(seq, max_len):
    padding = [vocab["<PAD>"]] * (max_len - len(seq))
    return padding + seq


padded_sequences = [pad_sequence(seq, max_sequence_len) for seq in input_sequences]

padded_sequences[:2]

[[0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  2,
  3,
  4],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  2,
  3,
  4,
  5]]

# identificamos X y Y

In [10]:
X = []
y = []

for seq in padded_sequences:
    X.append(seq[:-1])
    y.append(seq[-1])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

Forma de X: torch.Size([19566, 88])
Forma de y: torch.Size([19566])


# Crear Dataset y DataLoader en PyTorch
Dataset organiza los datos.

DataLoader permite entrenar por lotes o batches.

In [11]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


train_dataset = TextDataset(X, y)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

# Entrenar Word2Vec

In [12]:
embedding_dim = 100

word2vec_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=embedding_dim,
    window=5,
    min_count=1,
    workers=4
)

print("Word2Vec entrenado")

Word2Vec entrenado


In [13]:
word2vec_model.wv.most_similar("learning", topn=5)

[('help', 0.9606783986091614),
 ('will', 0.959150493144989),
 ('if', 0.9591254591941833),
 ('this', 0.9590442180633545),
 ('were', 0.9589937925338745)]

# Crear matriz de embeddings

In [14]:
embedding_matrix = np.random.normal(
    scale=0.6,
    size=(vocab_size, embedding_dim)
)

embedding_matrix[vocab["<PAD>"]] = np.zeros(embedding_dim)

for word, index in vocab.items():
    if word in word2vec_model.wv:
        embedding_matrix[index] = word2vec_model.wv[word]

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float32)

print("Forma de embedding_matrix:", embedding_matrix.shape)

Forma de embedding_matrix: torch.Size([6194, 100])


# Crear modelo LSTM con PyTorch

In [15]:
class NextWordLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(embedding_matrix)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)

        lstm_output, (hidden, cell) = self.lstm(embedded)

        last_hidden = hidden[-1]
        last_hidden = self.dropout(last_hidden)

        output = self.fc(last_hidden)

        return output


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hidden_dim = 128

model = NextWordLSTM(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    embedding_matrix=embedding_matrix
).to(device)

model

NextWordLSTM(
  (embedding): Embedding(6194, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=6194, bias=True)
)

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [18]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")


Epoch [1/10] - Loss: 7.5426
Epoch [2/10] - Loss: 7.0205
Epoch [3/10] - Loss: 6.8387
Epoch [4/10] - Loss: 6.6725
Epoch [5/10] - Loss: 6.5172
Epoch [6/10] - Loss: 6.3850
Epoch [7/10] - Loss: 6.2561
Epoch [8/10] - Loss: 6.1300
Epoch [9/10] - Loss: 5.9945
Epoch [10/10] - Loss: 5.8662


# Función para predecir la siguiente palabra

In [19]:
def predict_next_word(seed_text):
    model.eval()

    seed_text = clean_text(seed_text)
    sequence = text_to_sequence(seed_text)

    sequence = pad_sequence(sequence, max_sequence_len - 1)
    sequence = torch.tensor([sequence], dtype=torch.long).to(device)

    with torch.no_grad():
        output = model(sequence)
        predicted_index = torch.argmax(output, dim=1).item()

    return index_to_word.get(predicted_index, "<OOV>")

In [20]:
predict_next_word("machine learning is")

'be'

In [21]:
def complete_sentence(seed_text, next_words=10):
    result = seed_text

    for _ in range(next_words):
        next_word = predict_next_word(result)
        result += " " + next_word

    return result

In [22]:
complete_sentence("machine learning is", next_words=8)

'machine learning is be a united of the time of the'

In [23]:
complete_sentence("artificial intelligence can", next_words=8)

'artificial intelligence can a time of the time of the time'